# Road Extraction from Aerial Imagery — a 4-Model Comparative Study

Which segmentation architecture is best for extracting road networks from
aerial imagery? This project trained **U-Net, U-Net++, DeepLabV3+ and
LinkNet** under a *strictly identical* protocol (same ResNet-34 encoder,
loss, optimizer, schedule, augmentation, batch size and data) on the
Massachusetts Roads Dataset, so measured differences come from the
architecture alone.

## Test-set results (441 held-out tiles)

| Model | IoU | F1 | Precision | Recall | Params (M) | ms/tile |
|---|---|---|---|---|---|---|
| **U-Net++** | **0.654** | **0.791** | 0.808 | 0.774 | 26.1 | 199.8 |
| U-Net | 0.650 | 0.788 | 0.807 | 0.770 | 24.4 | 110.5 |
| LinkNet | 0.649 | 0.787 | 0.803 | 0.771 | **21.8** | **105.0** |
| DeepLabV3+ | 0.642 | 0.782 | 0.798 | 0.766 | 22.4 | 117.5 |
| Ensemble (x4) | 0.656 | 0.792 | 0.814 | 0.772 | 94.7 | 488.3 |

**Highlights**
- U-Net++ wins on accuracy; LinkNet delivers 99% of that accuracy at half
  the inference cost.
- Zero-shot transfer to DeepGlobe (India/Indonesia/Thailand) *inverts* the
  ranking: LinkNet generalizes best (IoU 0.159), U-Net++ worst (0.125).
- Geo-referenced output agrees with OpenStreetMap as well as the human
  ground truth does (correctness 0.985 vs 0.979).

🔗 [Source code](https://github.com/shuklaabhinavv/road-extraction) · [Live demo — extract roads anywhere on Earth](https://huggingface.co/spaces/shuklaabhinavv/roadx)

---
Below: load the trained weights and run road extraction on test images,
end-to-end, right here. *(Attach the `massachusetts-roads-dataset` by
balraj98 as input, enable Internet, then Run All — no GPU needed.)*

In [ ]:
%pip install -q segmentation-models-pytorch huggingface_hub


In [ ]:
import os; os.makedirs('roadx/data', exist_ok=True); os.makedirs('configs', exist_ok=True); open('roadx/__init__.py','w').close(); open('roadx/data/__init__.py','w').close()


In [ ]:
%%writefile roadx/data/dataset.py
from pathlib import Path

import albumentations as A
import numpy as np
import torch
from albumentations.pytorch import ToTensorV2
from PIL import Image
from torch.utils.data import Dataset

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def train_transform() -> A.Compose:
    return A.Compose(
        [
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ]
    )


def eval_transform() -> A.Compose:
    return A.Compose(
        [
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ]
    )


class RoadDataset(Dataset):
    """512x512 image tiles with binary road masks."""

    def __init__(self, root: Path, split: str, train: bool, limit: int | None = None):
        self.img_dir = Path(root) / split / "images"
        self.msk_dir = Path(root) / split / "masks"
        self.items = sorted(p.name for p in self.img_dir.glob("*.png"))
        if limit:
            self.items = self.items[:limit]
        if not self.items:
            raise FileNotFoundError(f"no tiles found in {self.img_dir}")
        self.tf = train_transform() if train else eval_transform()

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        name = self.items[idx]
        img = np.asarray(Image.open(self.img_dir / name).convert("RGB"))
        msk = (np.asarray(Image.open(self.msk_dir / name).convert("L")) > 127).astype(np.float32)
        out = self.tf(image=img, mask=msk)
        return out["image"], out["mask"].unsqueeze(0)


In [ ]:
%%writefile roadx/models.py
import segmentation_models_pytorch as smp
import torch

ARCHS = {
    "unet": smp.Unet,
    "unetpp": smp.UnetPlusPlus,
    "deeplabv3plus": smp.DeepLabV3Plus,
    "linknet": smp.Linknet,
}


def build_model(
    name: str, encoder: str = "resnet34", encoder_weights: str | None = "imagenet"
) -> torch.nn.Module:
    if name not in ARCHS:
        raise ValueError(f"unknown model '{name}', choose from {sorted(ARCHS)}")
    return ARCHS[name](
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=1,
    )


def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


In [ ]:
%%writefile roadx/predict.py
"""Run inference on a full-size satellite image via sliding window, save the
binary road mask and a red overlay visualization.

    python -m roadx.predict --checkpoint runs/unet/best.pt --image path/to.tiff --out results/
"""

import argparse
from pathlib import Path

import numpy as np
import torch
from PIL import Image

from roadx.data.dataset import IMAGENET_MEAN, IMAGENET_STD
from roadx.models import build_model, pick_device

TILE = 512


@torch.no_grad()
def predict_image(model: torch.nn.Module, img: np.ndarray, device: torch.device) -> np.ndarray:
    """Sliding-window prediction over an HxWx3 uint8 image -> HxW probability map."""
    h, w = img.shape[:2]
    ph = (h + TILE - 1) // TILE * TILE
    pw = (w + TILE - 1) // TILE * TILE
    padded = np.zeros((ph, pw, 3), dtype=np.uint8)
    padded[:h, :w] = img

    x = padded.astype(np.float32) / 255.0
    x = (x - np.array(IMAGENET_MEAN)) / np.array(IMAGENET_STD)
    x = torch.from_numpy(x.transpose(2, 0, 1)).float()

    prob = np.zeros((ph, pw), dtype=np.float32)
    for y0 in range(0, ph, TILE):
        for x0 in range(0, pw, TILE):
            patch = x[:, y0 : y0 + TILE, x0 : x0 + TILE].unsqueeze(0).to(device)
            logits = model(patch)
            prob[y0 : y0 + TILE, x0 : x0 + TILE] = torch.sigmoid(logits)[0, 0].cpu().numpy()
    return prob[:h, :w]


def overlay(img: np.ndarray, mask: np.ndarray, alpha: float = 0.55) -> np.ndarray:
    out = img.copy()
    red = np.zeros_like(img)
    red[..., 0] = 255
    sel = mask.astype(bool)
    out[sel] = (alpha * red[sel] + (1 - alpha) * img[sel]).astype(np.uint8)
    return out


def main() -> None:
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--checkpoint", type=Path, required=True)
    p.add_argument("--image", type=Path, required=True)
    p.add_argument("--out", type=Path, default=Path("results"))
    p.add_argument("--threshold", type=float, default=0.5)
    args = p.parse_args()

    device = pick_device()
    ckpt = torch.load(args.checkpoint, map_location="cpu")
    model = build_model(ckpt["model"], ckpt["encoder"], encoder_weights=None)
    model.load_state_dict(ckpt["state_dict"])
    model.to(device).eval()

    img = np.asarray(Image.open(args.image).convert("RGB"))
    prob = predict_image(model, img, device)
    mask = prob > args.threshold

    args.out.mkdir(parents=True, exist_ok=True)
    stem = args.image.stem
    Image.fromarray((mask * 255).astype(np.uint8)).save(args.out / f"{stem}_{ckpt['model']}_mask.png")
    Image.fromarray(overlay(img, mask)).save(args.out / f"{stem}_{ckpt['model']}_overlay.png")
    print(f"road pixels: {mask.mean():.2%} of image")
    print(f"wrote {args.out}/{stem}_{ckpt['model']}_mask.png and _overlay.png")


if __name__ == "__main__":
    main()


In [ ]:
# trained checkpoints are hosted in the project's Hugging Face Space
from huggingface_hub import hf_hub_download

CKPTS = {}
for name in ('unetpp', 'linknet'):
    CKPTS[name] = hf_hub_download('shuklaabhinavv/roadx', f'runs/{name}/best.pt', repo_type='space')
    print('downloaded', name)


In [ ]:
import os
import numpy as np
import torch
from PIL import Image
from roadx.models import build_model, pick_device
from roadx.predict import predict_image, overlay

device = pick_device()
models = {}
for name, path in CKPTS.items():
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    m = build_model(ckpt['model'], ckpt['encoder'], encoder_weights=None)
    m.load_state_dict(ckpt['state_dict'])
    m.to(device).eval()
    models[name] = m
print('loaded', list(models), 'on', device.type)


In [ ]:
# find test images in the attached public dataset (any folder layout)
test_imgs = []
for dirpath, _, files in os.walk('/kaggle/input'):
    if dirpath.endswith(('test', 'test_sat')) or '/test' in dirpath:
        test_imgs += [os.path.join(dirpath, f) for f in files
                      if f.endswith(('.tiff', '.tif', '.png', '.jpg')) and 'label' not in dirpath]
test_imgs = sorted(set(test_imgs))[:3]
assert test_imgs, 'attach the massachusetts-roads-dataset (balraj98) as input'
print(len(test_imgs), 'test images selected')


In [ ]:
import matplotlib.pyplot as plt

for path in test_imgs:
    img = np.asarray(Image.open(path).convert('RGB'))
    fig, axes = plt.subplots(1, 1 + len(models), figsize=(5.5 * (1 + len(models)), 5.5))
    axes[0].imshow(img); axes[0].set_title('input'); axes[0].axis('off')
    for ax, (name, m) in zip(axes[1:], models.items()):
        mask = predict_image(m, img, device) > 0.5
        ax.imshow(overlay(img, mask))
        ax.set_title(f'{name} — {mask.mean():.1%} road'); ax.axis('off')
    plt.tight_layout(); plt.show()


## How this was built
The full pipeline (data prep, identical-protocol training on Kaggle T4s,
evaluation, geo-referencing with OpenStreetMap validation, and the web app)
is open source: **[github.com/shuklaabhinavv/road-extraction](https://github.com/shuklaabhinavv/road-extraction)**.

Try the interactive demo — extract roads over your own city: **[huggingface.co/spaces/shuklaabhinavv/roadx](https://huggingface.co/spaces/shuklaabhinavv/roadx)**